# BioChannel Notebook 1 — Cell Decision Simulator

**Preprocess RNA-seq/single-cell data and generate cell-state decision outputs.**

This notebook is part of the **BioChannel** Kaggle hackathon project. It is designed to be run inside Kaggle Notebooks and then reused by the Streamlit dashboard.

## How to use this notebook in BioChannel

1. Attach the relevant Kaggle dataset using **Add Data** in the Kaggle notebook sidebar.
2. Run the notebook end-to-end.
3. Save processed outputs into `/kaggle/working/processed/`.
4. Copy the exported CSV/JSON files into the BioChannel Streamlit app under `data/processed/`.
5. Reuse the helper functions in the app modules: `data_loader.py`, `preprocessing.py`, `simulation.py`, `info_theory.py`, and `prediction.py`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mutual_info_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

np.random.seed(42)


In [ ]:
from pathlib import Path

INPUT_DIR = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
PROCESSED_DIR = WORKING_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

print('Input datasets available:')
for p in INPUT_DIR.glob('*'):
    print(' -', p)

## Dataset

Recommended datasets:

- Allen Institute single-cell RNA-seq: `allen-institute-for-cell-science/single-cell-rna-seq-data`
- Alternative RNA-seq cancer dataset: `uciml/gene-expression-cancer-rna-seq`

Science mapping:

`external signal → noisy pathway/channel → gene expression → cell decision`

Rows represent cells or samples. Columns represent gene-expression measurements. Clusters in expression space are treated as observable cell-state regions.

In [ ]:
def find_csv_files(root=INPUT_DIR):
    return list(root.rglob('*.csv'))

csv_files = find_csv_files()
print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:20]:
    print(f)

In [ ]:
def load_first_expression_like_csv(csv_files):
    for f in csv_files:
        try:
            df = pd.read_csv(f)
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) >= 20 and len(df) >= 20:
                print('Using:', f)
                return df
        except Exception:
            pass
    print('No suitable dataset found. Creating demo gene-expression-like data.')
    n_cells, n_genes = 350, 80
    states = np.random.choice(['Normal-like','Cancer-like','Stressed','Dormant','Resistant'], n_cells)
    X = np.random.normal(0, 1, (n_cells, n_genes))
    for i, s in enumerate(states):
        if s == 'Cancer-like': X[i, :15] += 1.4
        elif s == 'Stressed': X[i, 15:30] += 1.2
        elif s == 'Dormant': X[i, 30:45] += 1.1
        elif s == 'Resistant': X[i, 45:60] += 1.3
    df = pd.DataFrame(X, columns=[f'GENE_{i:03d}' for i in range(n_genes)])
    df['cell_state'] = states
    return df

df = load_first_expression_like_csv(csv_files)
df.head()

## Processing

The app needs a clean matrix `X`, optional labels, PCA coordinates, and a small set of high-variance genes. Use this output in BioChannel’s **Cell Decision Simulator** tab.

In [ ]:
# Keep numeric expression columns. Remove id-like columns if needed.
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Basic cleaning
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all').fillna(0)

# Optional log transform only when data are non-negative
if (numeric_df.min().min() >= 0):
    X = np.log1p(numeric_df)
else:
    X = numeric_df.copy()

# Select top variable genes/features for fast dashboard rendering
variances = X.var(axis=0).sort_values(ascending=False)
top_features = variances.head(min(500, len(variances))).index.tolist()
X_top = X[top_features]

X_scaled = StandardScaler().fit_transform(X_top)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

# Label inference for demo; replace with real labels if available
label_col = None
for c in df.columns:
    if c.lower() in ['label','class','diagnosis','cell_state','type']:
        label_col = c
        break
labels = df[label_col].astype(str).values if label_col else np.array(['Unknown'] * len(df))

processed = pd.DataFrame({'PC1': coords[:,0], 'PC2': coords[:,1], 'cell_state': labels})
processed.to_csv(PROCESSED_DIR / 'cell_decision_pca.csv', index=False)
pd.Series(top_features).to_csv(PROCESSED_DIR / 'cell_decision_top_features.csv', index=False, header=['feature'])

processed.head()

In [ ]:
plt.figure(figsize=(7,5))
for label in pd.unique(processed['cell_state'])[:8]:
    subset = processed[processed['cell_state'] == label]
    plt.scatter(subset['PC1'], subset['PC2'], s=12, alpha=0.7, label=str(label))
plt.title('BioChannel Cell-State Landscape')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(loc='best', fontsize=8)
plt.show()

## Simple decision model exported for the app

This model converts user-controlled inputs into predicted cell decision probabilities. It is intentionally lightweight for the hackathon MVP.

In [ ]:
def softmax(scores):
    x = np.array(scores, dtype=float)
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

def cell_decision(signal_strength=0.55, noise_level=0.25, stress_level=0.35, drug_pressure=0.30, resistance_pressure=0.25):
    proliferation = 1.2 * signal_strength - stress_level - drug_pressure - 0.4 * noise_level
    apoptosis = 1.2 * drug_pressure + 0.7 * stress_level - resistance_pressure
    dormancy = stress_level + 0.9 * noise_level - 0.25 * signal_strength
    resistance = resistance_pressure + 0.5 * drug_pressure + 0.6 * noise_level
    probs = softmax([proliferation, apoptosis, dormancy, resistance])
    return dict(zip(['Proliferation','Apoptosis','Dormancy','Resistance'], probs.round(4)))

example = cell_decision()
with open(PROCESSED_DIR / 'cell_decision_example_output.json', 'w') as f:
    json.dump(example, f, indent=2)
example

## Streamlit integration

Use these generated files:

- `cell_decision_pca.csv` → PCA scatter plot
- `cell_decision_top_features.csv` → feature/gene selector and explanation context
- `cell_decision_example_output.json` → sanity check for dashboard output

Gemma explanation prompt should include signal strength, noise level, stress, drug pressure, resistance pressure, predicted decision probabilities, and top genes/features.